# Magnetball online mode — Azure setup notebook

The steps in [`docs/SETUP.md`](docs/SETUP.md) as cells you run top to bottom, once, on your own
machine. Everything after this is done by a push to `main` through `.github/workflows/azure.yml`.

**Needs:** the Azure CLI (`winget install Microsoft.AzureCLI`, then reopen the terminal Jupyter runs
from). The GitHub CLI (`winget install GitHub.cli`, then `gh auth login`) is optional: with it, the
secret is written to the repo for you and the deploy is started; without it two steps are done by hand.

**The one secret this creates is never printed.** It goes to the clipboard (and to GitHub if `gh` is
there). Do not paste it into a cell, and clear this notebook's outputs before committing it (Kernel →
Restart & Clear Output).

In [ ]:
# One helper for every cell: run a command, stream its output, hand back stdout.
import subprocess, sys, json, os, platform

REPO = 'AmmarRica/magnetball'      # the GitHub repo the workflow lives in
SP_NAME = 'magnetball-deploy'      # the identity GitHub deploys as
SUBSCRIPTION = ''                  # leave blank to use whatever `az` picks, or paste an id from the table below

def run(cmd, check=True, quiet=False):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout and not quiet: print(r.stdout.rstrip())
    if r.stderr.strip(): print(r.stderr.rstrip(), file=sys.stderr)
    if check and r.returncode != 0: raise SystemExit(f'exit {r.returncode}: {cmd}')
    return r.stdout.strip()

# Jupyter inherits the PATH of the terminal it was started from, which need not include a CLI
# installed since. Put the usual install folders on it so a fresh install is seen without a restart.
for d in [r'C:\Program Files\Microsoft SDKs\Azure\CLI2\wbin', r'C:\Program Files (x86)\Microsoft SDKs\Azure\CLI2\wbin',
          r'C:\Program Files\GitHub CLI', os.path.expandvars(r'%LOCALAPPDATA%\Programs\GitHub CLI'),
          '/opt/homebrew/bin', '/usr/local/bin']:
    if os.path.isdir(d) and d not in os.environ['PATH']:
        os.environ['PATH'] = d + os.pathsep + os.environ['PATH']

def have(tool):
    return subprocess.run(f'{tool} --version', shell=True, capture_output=True).returncode == 0

print('az :', 'ok' if have('az') else 'MISSING — winget install Microsoft.AzureCLI; if it is installed, start Jupyter from a fresh terminal')
print('gh :', 'ok' if have('gh') else 'not installed (optional) — winget install GitHub.cli')

## 1. Log in to Azure
A browser window opens. If it fails with `AADSTS50076`, run the second cell with the tenant id from the error.

In [ ]:
run('az login --output none', check=False)
run('az account list --output table')

In [ ]:
# Only if the login above asked for multi-factor auth on a specific tenant:
# run('az login --tenant <tenant id from the error> --output none')

## 2. Pick the subscription
If the table above is empty, add one in the portal first (Subscriptions → Add → Free Trial or Pay-As-You-Go), then re-run step 1.

In [ ]:
if SUBSCRIPTION:
    run(f'az account set --subscription "{SUBSCRIPTION}"')
SUB_ID = run('az account show --query id -o tsv', quiet=True)
print('using subscription', SUB_ID, '—', run('az account show --query name -o tsv', quiet=True))

## 3. Register the five services (once)
Each takes about a minute in the background. "Registering" is normal; nothing waits on it.

In [ ]:
for ns in ['Microsoft.Web', 'Microsoft.SignalRService', 'Microsoft.Storage', 'Microsoft.App', 'Microsoft.OperationalInsights']:
    run(f'az provider register --namespace {ns} --output none')
run('az provider list --query "[?namespace==\'Microsoft.Web\' || namespace==\'Microsoft.SignalRService\' || namespace==\'Microsoft.Storage\' || namespace==\'Microsoft.App\' || namespace==\'Microsoft.OperationalInsights\'].{ns:namespace, state:registrationState}" --output table')

## 4. Create the deploy identity and put its secret where it belongs
The JSON `az` prints is a client secret (Contributor on the subscription, expires in a year). This cell
keeps it in a variable, copies it to the clipboard, and **does not print it**. Re-running is safe: the
same name gets a fresh secret and the old one stops working.

In [ ]:
creds = run(f'az ad sp create-for-rbac --name {SP_NAME} --role Contributor --scopes "/subscriptions/{SUB_ID}" --sdk-auth', quiet=True)
doc = json.loads(creds)
assert doc.get('clientId') and doc.get('tenantId'), 'unexpected output from create-for-rbac'
print('created', SP_NAME, '— clientId', doc['clientId'], '(secret not shown)')

clip = {'Windows': 'clip', 'Darwin': 'pbcopy'}.get(platform.system(), 'xclip -selection clipboard')
try:
    subprocess.run(clip, input=creds, text=True, shell=True, check=True)
    print('copied to the clipboard')
except Exception as e:
    print('could not copy to the clipboard:', e, '— run the next cell, or `print(creds)` here and copy by hand')

### 4b. Store it as the repo secret `AZURE_CREDENTIALS`
With the GitHub CLI logged in, this cell does it. Otherwise: the repo on GitHub → **Settings** →
**Secrets and variables** → **Actions** → **New repository secret**, name `AZURE_CREDENTIALS`,
value = paste (it is on your clipboard).

In [ ]:
if have('gh'):
    subprocess.run(f'gh secret set AZURE_CREDENTIALS --repo {REPO}', input=creds, text=True, shell=True, check=True)
    run(f'gh secret list --repo {REPO}')
else:
    print('gh not installed: paste the clipboard into the AZURE_CREDENTIALS repo secret by hand')

## 5. Start the first deploy and watch it
Four jobs, about six minutes. Without `gh`: the repo → **Actions** → *Deploy to Azure* → **Run workflow**.

In [ ]:
if have('gh'):
    run(f'gh workflow run azure.yml --repo {REPO} --ref main')
    import time; time.sleep(5)
    run(f'gh run list --repo {REPO} --workflow azure.yml --limit 1')
    print('watching the newest run (Ctrl-C / interrupt the kernel to stop watching; the run carries on)')
    subprocess.run(f'gh run watch --repo {REPO} --exit-status', shell=True)
else:
    print('start it from the Actions tab: Deploy to Azure → Run workflow')

## 6. Once, after the first successful run: make the image public
Azure pulls the match server image anonymously. On GitHub: your profile → **Packages** →
`magnetball-match` → **Package settings** → **Change visibility** → Public. Then re-run step 5.
There is no CLI for this; the cell below only checks whether it has been done.

In [ ]:
# ghcr.io answers EVERY bare request with 401, public image or not: a client first asks for an
# anonymous pull token and reads the tag list with it. Checked the bare way this printed
# "NO (HTTP 401)" for a package that was already public.
import urllib.request, urllib.error
owner = REPO.split('/')[0].lower()
try:
    tok = json.load(urllib.request.urlopen(f'https://ghcr.io/token?scope=repository:{owner}/magnetball-match:pull'))['token']
    req = urllib.request.Request(f'https://ghcr.io/v2/{owner}/magnetball-match/tags/list', headers={'Authorization': f'Bearer {tok}'})
    with urllib.request.urlopen(req) as r:
        print('public: yes —', json.load(r).get('tags', [])[:5])
except urllib.error.HTTPError as e:
    print('public: NO (HTTP', e.code, ') — change the package visibility on GitHub, then re-run the deploy')


## 7. Check each piece
The table from SETUP.md, in the order that makes a failure point at one thing. The first API call after a quiet spell cold-starts (2–3 s); the match server takes 10–20 s.

In [ ]:
import urllib.request, urllib.error
SITE = 'https://blue-glacier-0a03a2b0f.4.azurestaticapps.net'   # the addresses the last deploy step prints; edit if yours differ

def get(url, timeout=40):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            body = r.read().decode('utf-8', 'replace')
            return r.status, body[:200]
    except Exception as e:
        return None, str(e)

st, online = get(SITE + '/online.json')
print('site /online.json :', st, online)
cfg = json.loads(online) if st == 200 else {}
if cfg.get('api'):
    print('api  /api/scores  :', *get(cfg['api'] + '/api/scores'))
    print('api  room token   :', *get(cfg['api'] + '/api/room-token?room=ABCD'))
if cfg.get('match'):
    print('match /health     :', *get(cfg['match'] + '/health'))
else:
    print('match: online.json names no match server — hosted play is hidden in the game until it does')
print()
print('last: on two devices open', SITE, '→ Match → Game → Online, type ABCD, Play online.')

## Later
- **Rotate the key:** regenerate it in the portal, re-run the deploy; the Bicep reads the new one.
- **Tear everything down:** one resource group holds it all.

In [ ]:
# Uncomment to delete every Azure resource this setup created. Irreversible.
# run('az group delete --name magnetball-rg --yes --no-wait')